In [ ]:
import marimo as mo
import os
from pathlib import Path

import anthropic
from dotenv import load_dotenv
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index

from rag_helper import RAG

# Load .env from the repo root (one level up from this module's folder).
load_dotenv(Path(__file__).resolve().parent.parent / ".env")

QUERY = "How does the agentic loop keep calling the model until it stops?"
MODEL = os.environ.get("GLM_MODEL", "glm-4.6")
client = anthropic.Anthropic(
    api_key=os.environ["ZAI_API_KEY"],
    base_url=os.environ["ZAI_BASE_URL"],
)

# Module 1 homework: Agentic RAG

Knowledge base: the course's own lesson pages, fetched from
`DataTalksClub/llm-zoomcamp` at commit `8c1834d` via `gitsource`.

LLM: `glm-4.6` through Z.ai's Anthropic-compatible endpoint.

In [ ]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [f.parse() for f in reader.read()]
mo.md(f"**Q1.** How many lesson pages are in the dataset? **{len(documents)}**")

**Q1.** How many lesson pages are in the dataset? **72**

## Q2. Indexing and searching

In [ ]:
index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)
results = index.search(QUERY, num_results=5)
top = results[0]["filename"]
_body = "\n".join(
    [f"First result: `{top}`", "", "Top 5:", ""]
    + [f"- `{r['filename']}`" for r in results]
)
mo.md(_body)

First result: `01-agentic-rag/lessons/14-agentic-loop.md`

Top 5:

- `01-agentic-rag/lessons/14-agentic-loop.md`
- `01-agentic-rag/lessons/15-frameworks.md`
- `01-agentic-rag/lessons/13-function-calling.md`
- `01-agentic-rag/lessons/11-agents-intro.md`
- `01-agentic-rag/lessons/16-other-frameworks.md`

## Q3. RAG (input tokens)

In [ ]:
rag = RAG(index=index, llm_client=client, model=MODEL)
answer, usage = rag.rag(QUERY)
_body = "\n".join(
    [
        f"Input (prompt) tokens: **{usage.input_tokens}**",
        "",
        "Answer (start):",
        "",
        "```text",
        answer.strip(),
        "```",
    ]
)
mo.md(_body)

Input (prompt) tokens: **7200**

Answer (start):

```text
Based on the provided context, specifically the section **"The full agent loop"**, the agentic loop keeps calling the model using a `while` loop with the following logic:

1.  **A Continuous Loop:** It uses a `while True` loop that continues indefinitely.
2.  **Tracking Function Calls:** Inside the loop, a flag called `has_function_calls` is initialized to `False`. The code then checks the model's response items. If any item is a `function_call`, it executes the function, appends the result to the message history, and sets `has_function_calls` to `True`.
3.  **The Exit Condition:** At the end of each iteration, the loop checks the flag: `if has_function_calls == False: break`.

Essentially, the loop repeats as long as the model requests tools (function calls). It stops when the model returns a response without any function calls, meaning it has provided a final answer. The text states: *"The loop stops when the model returns a final answer with no more tool calls."*
```

## Q4. Chunking

In [ ]:
chunks = chunk_documents(documents, size=2000, step=1000)
mo.md(
    f"""
    Chunks with `size=2000, step=1000`: **{len(chunks)}**

    Chunk keys: `{list(chunks[0].keys())}`
    """
)

Chunks with `size=2000, step=1000`: **295**

Chunk keys: `['start', 'content', 'filename']`

## Q5. RAG with chunking (fewer input tokens)

In [ ]:
chunk_index = Index(text_fields=["content"], keyword_fields=["filename"])
chunk_index.fit(chunks)
chunk_rag = RAG(index=chunk_index, llm_client=client, model=MODEL)
_, chunk_usage = chunk_rag.rag(QUERY)
ratio = round(usage.input_tokens / chunk_usage.input_tokens, 1)
mo.md(
    f"""
    Chunked input tokens: **{chunk_usage.input_tokens}**
    (Q3 was {usage.input_tokens}, so **{ratio}x fewer**)
    """
)

Chunked input tokens: **2262**
(Q3 was 7200, so **3.2x fewer**)

## Q6. Turning it into an agent

The LLM gets a `search` tool over the chunk index and decides when
(and what) to search. Counting how many times it calls `search`.

In [ ]:
from toyaikit.chat.interface import IPythonChatInterface
from toyaikit.chat.runners import AnthropicMessagesRunner
from toyaikit.llm import AnthropicClient
from toyaikit.tools import Tools

calls = {"n": 0}

def search(query: str) -> str:
    """Search the LLM Zoomcamp lessons and return the most relevant passages."""
    calls["n"] += 1
    hits = chunk_index.search(query, num_results=5)
    return "\n\n".join(f"File: {h['filename']}\n{h['content'][:1500]}" for h in hits)

tools = Tools()
tools.add_tool(search)

llm = AnthropicClient(
    model=MODEL,
    api_key=os.environ["ZAI_API_KEY"],
    base_url=os.environ["ZAI_BASE_URL"],
    extra_kwargs={"max_tokens": 4096},
)
runner = AnthropicMessagesRunner(
    tools=tools,
    developer_prompt=(
        "You're a course teaching assistant. Answer the student's question "
        "using the search tool. Make multiple searches with different "
        "keywords before answering."
    ),
    chat_interface=IPythonChatInterface(),
    llm_client=llm,
)
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?"
)
_body = "\n".join(
    [
        f"The agent called `search` **{calls['n']}** times.",
        "",
        "Final answer (start):",
        "",
        "```text",
        str(result.last_message).strip(),
        "```",
    ]
)
mo.md(_body)

The agent called `search` **5** times.

Final answer (start):

```text
Based on the course materials, I can explain how the agentic loop works and how it differs from plain RAG.

## How the Agentic Loop Works

The agentic loop puts the **LLM in the driver's seat** - it decides what actions to take and in what order. Here's the anatomy of an agent:

### Three Core Components:

1. **Instructions** - The role and behavior (passed as a `developer` message)
2. **Tools** - Functions the agent can call (like `search`)
3. **Memory** - The message history containing every prompt, model output, and tool result

### The Loop Pattern:

The agentic loop follows this pattern:
```python
while True:
    1. Send messages to LLM
    2. Check if LLM wants to call a tool
    3. If yes: execute the tool, add result to messages, repeat
    4. If no: LLM gives final answer, stop
```

### Key Advantages:

- **Flexibility**: The LLM can fix typos, search multiple times, ask clarifying questions
- **Self-correction**: If a search returns poor results, the agent retries with different terms
- **Dynamic decision-making**: The LLM chooses what to do at each step

Example from the course: When asked about "Olama" (typo), the agent searches, finds poor results, then automatically retries with "Ollama" and recovers.

---

## Plain RAG (Comparison)

Plain RAG follows a **fixed, rigid flow**:

```python
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)
```

### Characteristics:
- **Always 3 steps**: Search → Build prompt → LLM answer
- **No flexibility**: Search runs once; if it returns garbage, the LLM gets garbage
- **No recovery**: Typos or unusual queries result in failures
- **Single-shot**: One API call per request

---

## Key Differences at a Glance

| Aspect | Plain RAG | Agentic Loop |
|--------|-----------|--------------|
| **Control** | Fixed pipeline | LLM decides what to do |
| **Flexibility** | Rigid - always 3 steps | Dynamic - varies per request |
| **Recovery** | None | Can retry, fix typos, re-search |
| **API Calls** | 1 per request | Multiple (until satisfied) |
| **Cost** | Lower | Higher (each loop iteration costs) |
| **Latency** | Lower | Higher (multiple round-trips) |
| **Predictability** | High | Lower (LLM chooses the path) |

---

## When to Use Which?

The course advises: **Use plain RAG when you can**. Agents add:
- More API calls → higher cost
- Higher latency
- More moving parts to monitor
- Less predictable behavior

Reach for agents when plain RAG isn't sufficient - for example, when you need multi-step reasoning, self-correction, or the ability to handle ambiguous queries.
```